In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PREPARED_DIR = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\prepared"
)

time_data = pd.read_csv(
    PREPARED_DIR / "time_tax_sdg_matched.csv.gz",
    keep_default_na=False,
    na_values=[""],
    dtype={
        "iso3": "string",
        "country": "string",
        "source": "string",
        "year": "Int64",
        "cross_source_conflict": "boolean",
    },
)

required = [
    "iso3",
    "country",
    "year",
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
    "cross_source_conflict",
]

if time_data[required].isna().any().any():
    raise ValueError("Missing values in required time-use fields.")

if not time_data["source"].eq("sdg").all():
    raise ValueError("Expected SDG time-use observations only.")

if time_data.duplicated(["iso3", "year"]).any():
    raise ValueError("Duplicate country-year observations.")

expected_gap = (
    time_data["female_unpaid_hours"]
    - time_data["male_unpaid_hours"]
)

if not np.allclose(
    time_data["time_gap_hours"],
    expected_gap,
    rtol=0,
    atol=1e-8,
):
    raise ValueError("Time-gap calculation is inconsistent.")

time_data = time_data.sort_values(
    ["iso3", "year"]
).reset_index(drop=True)

print(f"Observations: {len(time_data)}")
print(f"Entities: {time_data['iso3'].nunique()}")
print(f"Years: {time_data['year'].min()}–{time_data['year'].max()}")

Observations: 177
Entities: 89
Years: 2000–2019


In [2]:
entity_history = (
    time_data.groupby(["iso3", "country"])
    .agg(
        observations=("year", "size"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        observed_years=(
            "year",
            lambda values: ", ".join(
                str(int(year)) for year in sorted(values)
            ),
        ),
        flagged_observations=("cross_source_conflict", "sum"),
    )
    .reset_index()
)

entity_history["year_span"] = (
    entity_history["last_year"] - entity_history["first_year"]
)

entity_history["possible_adjacent_pairs"] = (
    entity_history["observations"] - 1
)

repeated_entities = (
    entity_history.loc[entity_history["observations"].ge(2)]
    .sort_values(
        ["observations", "country"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(f"Entities with repeated observations: {len(repeated_entities)}")
print(
    "Entities with one observation:",
    int(entity_history["observations"].eq(1).sum()),
)
print(
    "Possible adjacent-year pairs:",
    int(repeated_entities["possible_adjacent_pairs"].sum()),
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 120,
    "display.max_columns", None,
):
    display(repeated_entities)

Entities with repeated observations: 45
Entities with one observation: 44
Possible adjacent-year pairs: 88


,iso3,country,observations,first_year,last_year,observed_years,flagged_observations,year_span,possible_adjacent_pairs
0,USA,United States,17,2003,2019,"2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019",0,16,16
1,CHE,Switzerland,6,2000,2016,"2000, 2004, 2007, 2010, 2013, 2016",0,16,5
2,KAZ,Kazakhstan,5,2000,2018,"2000, 2003, 2006, 2012, 2018",0,18,4
3,MNG,Mongolia,5,2000,2019,"2000, 2007, 2011, 2015, 2019",0,19,4
4,JPN,Japan,4,2001,2016,"2001, 2006, 2011, 2016",0,15,3
5,KOR,"Korea, Rep.",4,2004,2019,"2004, 2009, 2014, 2019",0,15,3
6,KGZ,Kyrgyz Republic,4,2000,2015,"2000, 2005, 2010, 2015",0,15,3
7,LAO,Lao PDR,4,2003,2017,"2003, 2008, 2013, 2017",0,14,3
8,MEX,Mexico,4,2002,2019,"2002, 2009, 2014, 2019",0,17,3
9,NLD,Netherlands,4,2001,2012,"2001, 2003, 2006, 2012",0,11,3


In [3]:
pair_fields = [
    "year",
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
    "cross_source_conflict",
]

grouped = time_data.groupby("iso3", sort=False)

previous = (
    grouped[pair_fields]
    .shift(1)
    .add_suffix("_start")
)

current = (
    time_data[["iso3", "country"] + pair_fields]
    .rename(columns={
        column: f"{column}_end"
        for column in pair_fields
    })
)

adjacent_pairs = pd.concat([current, previous], axis=1)
adjacent_pairs = adjacent_pairs.loc[
    adjacent_pairs["year_start"].notna()
].copy()

adjacent_pairs["year_start"] = adjacent_pairs["year_start"].astype("Int64")
adjacent_pairs["year_end"] = adjacent_pairs["year_end"].astype("Int64")

adjacent_pairs["interval_years"] = (
    adjacent_pairs["year_end"] - adjacent_pairs["year_start"]
)

if adjacent_pairs["interval_years"].le(0).any():
    raise ValueError("Nonpositive observation interval.")

adjacent_pairs["either_endpoint_flagged"] = (
    adjacent_pairs["cross_source_conflict_start"].astype("boolean")
    | adjacent_pairs["cross_source_conflict_end"].astype("boolean")
)

# Absence of a source conflict does not establish survey comparability.
adjacent_pairs["survey_comparability"] = "Not yet assessed"

pair_inventory = adjacent_pairs[
    [
        "iso3",
        "country",
        "year_start",
        "year_end",
        "interval_years",
        "either_endpoint_flagged",
        "survey_comparability",
    ]
].copy()

print(f"Adjacent pairs: {len(pair_inventory)}")
print(f"Entities contributing pairs: {pair_inventory['iso3'].nunique()}")
print(
    "Pairs with a flagged endpoint:",
    int(pair_inventory["either_endpoint_flagged"].sum()),
)

print("\nObservation interval distribution")
display(
    pair_inventory.groupby("interval_years")
    .agg(
        pairs=("iso3", "size"),
        entities=("iso3", "nunique"),
    )
    .reset_index()
)

print("\nPair inventory")
with pd.option_context("display.max_rows", None):
    display(pair_inventory.reset_index(drop=True))

Adjacent pairs: 88
Entities contributing pairs: 45
Pairs with a flagged endpoint: 5

Observation interval distribution


,interval_years,pairs,entities
0,1,16,1
1,2,1,1
2,3,10,6
3,4,8,6
4,5,19,11
5,6,8,7
6,7,5,5
7,8,6,6
8,9,2,2
9,10,7,7



Pair inventory


,iso3,country,year_start,year_end,interval_years,either_endpoint_flagged,survey_comparability
0,ARG,Argentina,2005,2010,5,False,Not yet assessed
1,ARG,Argentina,2010,2013,3,False,Not yet assessed
2,ARM,Armenia,2004,2008,4,False,Not yet assessed
3,BEL,Belgium,2000,2005,5,False,Not yet assessed
4,BEL,Belgium,2005,2013,8,False,Not yet assessed
5,BGR,Bulgaria,2002,2010,8,False,Not yet assessed
6,BRA,Brazil,2009,2012,3,False,Not yet assessed
7,BRA,Brazil,2012,2017,5,True,Not yet assessed
8,BTN,Bhutan,2007,2015,8,False,Not yet assessed
9,CAN,Canada,2005,2010,5,True,Not yet assessed


In [4]:
changes = adjacent_pairs.copy()

TIME_CHANGE_COLUMNS = {
    "female_unpaid_hours": "delta_female_hours",
    "male_unpaid_hours": "delta_male_hours",
    "time_gap_hours": "delta_time_gap_hours",
}

for original, delta in TIME_CHANGE_COLUMNS.items():
    changes[delta] = (
        changes[f"{original}_end"]
        - changes[f"{original}_start"]
    )

if not np.allclose(
    changes["delta_time_gap_hours"].to_numpy(dtype=float),
    (
        changes["delta_female_hours"]
        - changes["delta_male_hours"]
    ).to_numpy(dtype=float),
    rtol=0,
    atol=1e-8,
):
    raise ValueError("Time-gap changes do not match their components.")

labor_levels = time_data[
    [
        "iso3",
        "year",
        "gender__SL.TLF.CACT.FE.ZS",
        "gender__SL.TLF.CACT.MA.ZS",
    ]
].rename(columns={
    "gender__SL.TLF.CACT.FE.ZS": "female_lfpr_pct",
    "gender__SL.TLF.CACT.MA.ZS": "male_lfpr_pct",
})

labor_levels["lfpr_gap_pp"] = (
    labor_levels["male_lfpr_pct"] - labor_levels["female_lfpr_pct"]
)

LABOR_COLUMNS = [
    "female_lfpr_pct",
    "male_lfpr_pct",
    "lfpr_gap_pp",
]

for endpoint in ["start", "end"]:
    endpoint_data = labor_levels.rename(columns={
        "year": f"year_{endpoint}",
        **{
            column: f"{column}_{endpoint}"
            for column in LABOR_COLUMNS
        },
    })

    changes = changes.merge(
        endpoint_data,
        on=["iso3", f"year_{endpoint}"],
        how="left",
        validate="many_to_one",
    )

changes["delta_female_lfpr_pp"] = (
    changes["female_lfpr_pct_end"] - changes["female_lfpr_pct_start"]
)
changes["delta_lfpr_gap_pp"] = (
    changes["lfpr_gap_pp_end"] - changes["lfpr_gap_pp_start"]
)

CHANGE_COLUMNS = [
    "delta_female_hours",
    "delta_male_hours",
    "delta_time_gap_hours",
    "delta_female_lfpr_pp",
    "delta_lfpr_gap_pp",
]

print(f"Pairs retained: {len(changes)}")
print("\nMissing change values")
display(changes[CHANGE_COLUMNS].isna().sum().to_frame("missing"))

display(
    changes[
        ["country", "year_start", "year_end", "interval_years"]
        + CHANGE_COLUMNS
        + ["either_endpoint_flagged"]
    ].head(10).round(3)
)

Pairs retained: 88

Missing change values


,missing
delta_female_hours,0
delta_male_hours,0
delta_time_gap_hours,0
delta_female_lfpr_pp,0
delta_lfpr_gap_pp,0


,country,year_start,year_end,interval_years,delta_female_hours,delta_male_hours,delta_time_gap_hours,delta_female_lfpr_pp,delta_lfpr_gap_pp,either_endpoint_flagged
0,Argentina,2005,2010,5,0.183,0.117,0.067,-1.409,0.095,False
1,Argentina,2010,2013,3,1.420,0.636,0.784,-0.047,-0.416,False
2,Armenia,2004,2008,4,-0.569,-0.051,-0.518,-0.558,0.201,False
3,Belgium,2000,2005,5,-0.250,0.000,-0.250,2.071,-2.130,False
4,Belgium,2005,2013,8,0.217,0.200,0.017,2.117,-3.762,False
5,Bulgaria,2002,2010,8,-0.583,-0.400,-0.183,1.622,1.912,False
6,Brazil,2009,2012,3,-1.069,-0.705,-0.364,-3.694,1.458,False
7,Brazil,2012,2017,5,-0.378,0.504,-0.882,1.480,-2.750,True
8,Bhutan,2007,2015,8,0.417,0.200,0.217,-9.831,6.193,False
9,Canada,2005,2010,5,0.000,0.100,-0.100,0.875,-1.779,True


In [5]:
CHANGE_TOLERANCE = 1e-6

change_views = {
    "All adjacent pairs": changes,
    "Exclude source conflicts": changes.loc[
        ~changes["either_endpoint_flagged"]
    ].copy(),
    "Exclude United States": changes.loc[
        changes["iso3"].ne("USA")
    ].copy(),
    "Exclude conflicts and United States": changes.loc[
        ~changes["either_endpoint_flagged"]
        & changes["iso3"].ne("USA")
    ].copy(),
}


def direction_labels(values):
    labels = pd.Series("Missing", index=values.index, dtype="string")
    labels.loc[values.notna()] = "Approximately unchanged"
    labels.loc[values.gt(CHANGE_TOLERANCE)] = "Increased"
    labels.loc[values.lt(-CHANGE_TOLERANCE)] = "Decreased"
    return labels


direction_rows = []
decomposition_rows = []

for view_name, frame in change_views.items():
    for variable in CHANGE_COLUMNS:
        counts = direction_labels(frame[variable]).value_counts()

        direction_rows.append({
            "view": view_name,
            "variable": variable,
            "increased": int(counts.get("Increased", 0)),
            "decreased": int(counts.get("Decreased", 0)),
            "approximately_unchanged": int(
                counts.get("Approximately unchanged", 0)
            ),
            "missing": int(counts.get("Missing", 0)),
        })

    narrowing = frame.loc[
        frame["delta_time_gap_hours"].lt(-CHANGE_TOLERANCE)
    ].copy()

    female_down = narrowing["delta_female_hours"].lt(-CHANGE_TOLERANCE)
    male_up = narrowing["delta_male_hours"].gt(CHANGE_TOLERANCE)

    decomposition_rows.append({
        "view": view_name,
        "narrowing_pairs": len(narrowing),
        "female_down_and_male_up": int((female_down & male_up).sum()),
        "female_down_without_male_up": int(
            (female_down & ~male_up).sum()
        ),
        "male_up_without_female_down": int(
            (~female_down & male_up).sum()
        ),
        "other_small_changes": int(
            (~female_down & ~male_up).sum()
        ),
    })

change_directions = pd.DataFrame(direction_rows)
narrowing_decomposition = pd.DataFrame(decomposition_rows)

print("Direction of observed changes")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(change_directions)

print("\nComponents of a narrowing time gap")
display(narrowing_decomposition)

for view_name in [
    "All adjacent pairs",
    "Exclude conflicts and United States",
]:
    frame = change_views[view_name]

    print(f"\nJoint directions: {view_name}")
    display(pd.crosstab(
        direction_labels(frame["delta_time_gap_hours"]),
        direction_labels(frame["delta_lfpr_gap_pp"]),
        rownames=["Time-gap change"],
        colnames=["Labor-participation-gap change"],
    ))

Direction of observed changes


,view,variable,increased,decreased,approximately_unchanged,missing
0,All adjacent pairs,delta_female_hours,27,59,2,0
1,All adjacent pairs,delta_male_hours,48,38,2,0
2,All adjacent pairs,delta_time_gap_hours,21,65,2,0
3,All adjacent pairs,delta_female_lfpr_pp,47,41,0,0
4,All adjacent pairs,delta_lfpr_gap_pp,27,61,0,0
5,Exclude source conflicts,delta_female_hours,27,55,1,0
6,Exclude source conflicts,delta_male_hours,46,35,2,0
7,Exclude source conflicts,delta_time_gap_hours,21,60,2,0
8,Exclude source conflicts,delta_female_lfpr_pp,45,38,0,0
9,Exclude source conflicts,delta_lfpr_gap_pp,27,56,0,0



Components of a narrowing time gap


,view,narrowing_pairs,female_down_and_male_up,female_down_without_male_up,male_up_without_female_down,other_small_changes
0,All adjacent pairs,65,24,27,14,0
1,Exclude source conflicts,60,23,24,13,0
2,Exclude United States,57,20,25,12,0
3,Exclude conflicts and United States,52,19,22,11,0



Joint directions: All adjacent pairs


Labor-participation-gap change,Decreased,Increased
Time-gap change,,
Approximately unchanged,2,0
Decreased,50,15
Increased,9,12



Joint directions: Exclude conflicts and United States


Labor-participation-gap change,Decreased,Increased
Time-gap change,,
Decreased,39,13
Increased,7,8


In [6]:
def change_correlation(frame, x_column, y_column, method):
    pair = (
        frame[["iso3", x_column, y_column]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[x_column, y_column])
    )

    x = pair[x_column]
    y = pair[y_column]

    coefficient = np.nan

    if len(pair) >= 3 and x.nunique() > 1 and y.nunique() > 1:
        if method == "pearson":
            coefficient = x.corr(y)
        elif method == "spearman":
            coefficient = x.rank().corr(y.rank())
        else:
            raise ValueError(f"Unknown method: {method}")

    return coefficient, len(pair), pair["iso3"].nunique()


correlation_rows = []

for view_name, frame in change_views.items():
    interval_band = pd.cut(
        frame["interval_years"],
        bins=[0, 1, 5, np.inf],
        labels=["1 year", "2–5 years", "6+ years"],
        right=True,
    )

    interval_views = {"All intervals": frame}

    for band in ["1 year", "2–5 years", "6+ years"]:
        subset = frame.loc[interval_band.eq(band)]
        if not subset.empty:
            interval_views[band] = subset

    for interval_name, subset in interval_views.items():
        for outcome in [
            "delta_female_lfpr_pp",
            "delta_lfpr_gap_pp",
        ]:
            pearson, pairs, entities = change_correlation(
                subset, "delta_time_gap_hours", outcome, "pearson"
            )
            spearman, _, _ = change_correlation(
                subset, "delta_time_gap_hours", outcome, "spearman"
            )

            correlation_rows.append({
                "view": view_name,
                "interval": interval_name,
                "related_change": outcome,
                "pairs": pairs,
                "entities": entities,
                "pearson_r": pearson,
                "spearman_rho": spearman,
            })

change_correlations = pd.DataFrame(correlation_rows)

for view_name in change_views:
    print(f"\nChange correlations: {view_name}")

    with pd.option_context("display.max_columns", None):
        display(
            change_correlations.loc[
                change_correlations["view"].eq(view_name)
            ]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )


Change correlations: All adjacent pairs


,interval,related_change,pairs,entities,pearson_r,spearman_rho
0,All intervals,delta_female_lfpr_pp,88,45,-0.160,-0.241
1,All intervals,delta_lfpr_gap_pp,88,45,0.225,0.315
2,1 year,delta_female_lfpr_pp,16,1,-0.064,-0.177
3,1 year,delta_lfpr_gap_pp,16,1,0.420,0.562
4,2–5 years,delta_female_lfpr_pp,38,19,0.170,0.044
5,2–5 years,delta_lfpr_gap_pp,38,19,0.055,0.127
6,6+ years,delta_female_lfpr_pp,34,33,-0.433,-0.492
7,6+ years,delta_lfpr_gap_pp,34,33,0.367,0.345



Change correlations: Exclude source conflicts


,interval,related_change,pairs,entities,pearson_r,spearman_rho
0,All intervals,delta_female_lfpr_pp,83,42,-0.240,-0.271
1,All intervals,delta_lfpr_gap_pp,83,42,0.289,0.309
2,1 year,delta_female_lfpr_pp,16,1,-0.064,-0.177
3,1 year,delta_lfpr_gap_pp,16,1,0.420,0.562
4,2–5 years,delta_female_lfpr_pp,35,17,0.195,0.030
5,2–5 years,delta_lfpr_gap_pp,35,17,-0.002,0.083
6,6+ years,delta_female_lfpr_pp,32,31,-0.424,-0.463
7,6+ years,delta_lfpr_gap_pp,32,31,0.365,0.330



Change correlations: Exclude United States


,interval,related_change,pairs,entities,pearson_r,spearman_rho
0,All intervals,delta_female_lfpr_pp,72,44,-0.143,-0.237
1,All intervals,delta_lfpr_gap_pp,72,44,0.195,0.246
2,2–5 years,delta_female_lfpr_pp,38,19,0.170,0.044
3,2–5 years,delta_lfpr_gap_pp,38,19,0.055,0.127
4,6+ years,delta_female_lfpr_pp,34,33,-0.433,-0.492
5,6+ years,delta_lfpr_gap_pp,34,33,0.367,0.345



Change correlations: Exclude conflicts and United States


,interval,related_change,pairs,entities,pearson_r,spearman_rho
0,All intervals,delta_female_lfpr_pp,67,41,-0.223,-0.263
1,All intervals,delta_lfpr_gap_pp,67,41,0.260,0.254
2,2–5 years,delta_female_lfpr_pp,35,17,0.195,0.030
3,2–5 years,delta_lfpr_gap_pp,35,17,-0.002,0.083
4,6+ years,delta_female_lfpr_pp,32,31,-0.424,-0.463
5,6+ years,delta_lfpr_gap_pp,32,31,0.365,0.330


In [7]:
ordered = time_data.sort_values(["iso3", "year"]).copy()

ordered["female_lfpr_pct"] = ordered["gender__SL.TLF.CACT.FE.ZS"]
ordered["male_lfpr_pct"] = ordered["gender__SL.TLF.CACT.MA.ZS"]
ordered["lfpr_gap_pp"] = (
    ordered["male_lfpr_pct"] - ordered["female_lfpr_pct"]
)

counts = ordered.groupby("iso3").size()
repeated_ids = counts.loc[counts.ge(2)].index

repeated = ordered.loc[ordered["iso3"].isin(repeated_ids)]

endpoint_columns = [
    "year",
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
    "female_lfpr_pct",
    "lfpr_gap_pp",
    "cross_source_conflict",
]

first = (
    repeated.drop_duplicates("iso3", keep="first")
    [["iso3", "country"] + endpoint_columns]
    .rename(columns={
        column: f"{column}_start"
        for column in endpoint_columns
    })
)

last = (
    repeated.drop_duplicates("iso3", keep="last")
    [["iso3"] + endpoint_columns]
    .rename(columns={
        column: f"{column}_end"
        for column in endpoint_columns
    })
)

first_last = first.merge(
    last,
    on="iso3",
    how="inner",
    validate="one_to_one",
)

first_last["interval_years"] = (
    first_last["year_end"] - first_last["year_start"]
)

first_last["either_endpoint_flagged"] = (
    first_last["cross_source_conflict_start"]
    | first_last["cross_source_conflict_end"]
)

CHANGE_MAPPING = {
    "female_unpaid_hours": "delta_female_hours",
    "male_unpaid_hours": "delta_male_hours",
    "time_gap_hours": "delta_time_gap_hours",
    "female_lfpr_pct": "delta_female_lfpr_pp",
    "lfpr_gap_pp": "delta_lfpr_gap_pp",
}

for original, delta in CHANGE_MAPPING.items():
    first_last[delta] = (
        first_last[f"{original}_end"]
        - first_last[f"{original}_start"]
    )

first_last["survey_comparability"] = "Not yet assessed"

print(f"First-to-last pairs: {len(first_last)}")
print(
    "Pairs with flagged endpoints:",
    int(first_last["either_endpoint_flagged"].sum()),
)

display(
    first_last["interval_years"]
    .describe()
    .to_frame("observation_span_years")
    .round(2)
)

First-to-last pairs: 45
Pairs with flagged endpoints: 4


,observation_span_years
count,45.0
mean,10.44
std,3.88
min,3.0
25%,8.0
50%,10.0
75%,13.0
max,19.0


In [8]:
first_last_views = {
    "All first-to-last pairs": first_last,
    "Exclude source conflicts": first_last.loc[
        ~first_last["either_endpoint_flagged"]
    ].copy(),
    "Exclude conflicts and United States": first_last.loc[
        ~first_last["either_endpoint_flagged"]
        & first_last["iso3"].ne("USA")
    ].copy(),
}

first_last_correlation_rows = []

for view_name, frame in first_last_views.items():
    for outcome in ["delta_female_lfpr_pp", "delta_lfpr_gap_pp"]:
        pearson, pairs, entities = change_correlation(
            frame, "delta_time_gap_hours", outcome, "pearson"
        )
        spearman, _, _ = change_correlation(
            frame, "delta_time_gap_hours", outcome, "spearman"
        )

        first_last_correlation_rows.append({
            "view": view_name,
            "related_change": outcome,
            "pairs": pairs,
            "entities": entities,
            "pearson_r": pearson,
            "spearman_rho": spearman,
        })

first_last_correlations = pd.DataFrame(first_last_correlation_rows)

print("First-to-last change correlations")
with pd.option_context("display.max_columns", None):
    display(first_last_correlations.round(3))

# Keep flagged cases visible in the full data, but use unflagged
# endpoint pairs for the initial case-review list.
case_review = first_last.loc[
    ~first_last["either_endpoint_flagged"]
    & first_last["delta_time_gap_hours"].lt(-CHANGE_TOLERANCE)
].copy()

case_review["case_type"] = "Participation gap approximately unchanged"

case_review.loc[
    case_review["delta_lfpr_gap_pp"].lt(-CHANGE_TOLERANCE),
    "case_type",
] = "Both gaps narrowed"

case_review.loc[
    case_review["delta_lfpr_gap_pp"].gt(CHANGE_TOLERANCE),
    "case_type",
] = "Time gap narrowed; participation gap widened"

female_down = case_review["delta_female_hours"].lt(-CHANGE_TOLERANCE)
male_up = case_review["delta_male_hours"].gt(CHANGE_TOLERANCE)

case_review["time_change_pattern"] = "Other small changes"

case_review.loc[
    female_down & male_up, "time_change_pattern"
] = "Female hours down; male hours up"

case_review.loc[
    female_down & ~male_up, "time_change_pattern"
] = "Female hours down; male hours not up"

case_review.loc[
    ~female_down & male_up, "time_change_pattern"
] = "Male hours up; female hours not down"

case_columns = [
    "country",
    "year_start",
    "year_end",
    "interval_years",
    "delta_female_hours",
    "delta_male_hours",
    "delta_time_gap_hours",
    "delta_female_lfpr_pp",
    "delta_lfpr_gap_pp",
    "time_change_pattern",
    "survey_comparability",
]

print("\nCase counts")
display(case_review["case_type"].value_counts().to_frame("entities"))

for case_type, group in case_review.groupby("case_type"):
    print(f"\n{case_type}")

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
    ):
        display(
            group.sort_values("country")[case_columns]
            .reset_index(drop=True)
            .round(3)
        )

First-to-last change correlations


,view,related_change,pairs,entities,pearson_r,spearman_rho
0,All first-to-last pairs,delta_female_lfpr_pp,45,45,-0.034,-0.207
1,All first-to-last pairs,delta_lfpr_gap_pp,45,45,0.118,0.213
2,Exclude source conflicts,delta_female_lfpr_pp,41,41,-0.117,-0.293
3,Exclude source conflicts,delta_lfpr_gap_pp,41,41,0.199,0.250
4,Exclude conflicts and United States,delta_female_lfpr_pp,40,40,-0.113,-0.282
5,Exclude conflicts and United States,delta_lfpr_gap_pp,40,40,0.200,0.253



Case counts


,entities
case_type,
Both gaps narrowed,25
Time gap narrowed; participation gap widened,10



Both gaps narrowed


,country,year_start,year_end,interval_years,delta_female_hours,delta_male_hours,delta_time_gap_hours,delta_female_lfpr_pp,delta_lfpr_gap_pp,time_change_pattern,survey_comparability
0,Belgium,2000,2013,13,-0.033,0.200,-0.233,4.188,-5.892,Female hours down; male hours up,Not yet assessed
1,Cuba,2001,2016,15,1.123,1.710,-0.586,7.765,-11.738,Male hours up; female hours not down,Not yet assessed
2,Denmark,2001,2009,8,0.250,0.267,-0.017,-0.106,-1.686,Male hours up; female hours not down,Not yet assessed
3,El Salvador,2010,2017,7,-0.608,-0.196,-0.413,-0.812,-0.279,Female hours down; male hours not up,Not yet assessed
4,Estonia,2000,2010,10,-0.650,-0.100,-0.550,2.494,-2.009,Female hours down; male hours not up,Not yet assessed
5,Finland,2000,2010,10,-0.148,0.113,-0.261,-0.695,-2.124,Female hours down; male hours up,Not yet assessed
6,Germany,2002,2013,11,-0.306,-0.019,-0.287,4.614,-4.518,Female hours down; male hours not up,Not yet assessed
7,"Hong Kong SAR, China",2002,2013,11,-0.700,-0.300,-0.400,2.420,-5.922,Female hours down; male hours not up,Not yet assessed
8,Italy,2003,2014,11,-0.360,0.264,-0.624,2.943,-5.933,Female hours down; male hours up,Not yet assessed
9,Japan,2001,2016,15,-0.117,0.200,-0.317,1.157,-6.358,Female hours down; male hours up,Not yet assessed



Time gap narrowed; participation gap widened


,country,year_start,year_end,interval_years,delta_female_hours,delta_male_hours,delta_time_gap_hours,delta_female_lfpr_pp,delta_lfpr_gap_pp,time_change_pattern,survey_comparability
0,Armenia,2004,2008,4,-0.569,-0.051,-0.518,-0.558,0.201,Female hours down; male hours not up,Not yet assessed
1,Bulgaria,2002,2010,8,-0.583,-0.400,-0.183,1.622,1.912,Female hours down; male hours not up,Not yet assessed
2,Kazakhstan,2000,2018,18,-2.100,-1.817,-0.283,-1.016,0.275,Female hours down; male hours not up,Not yet assessed
3,Kyrgyz Republic,2000,2015,15,-2.617,-1.033,-1.583,-10.427,5.063,Female hours down; male hours not up,Not yet assessed
4,Lao PDR,2003,2017,14,0.865,1.815,-0.950,-2.162,0.845,Male hours up; female hours not down,Not yet assessed
5,North Macedonia,2004,2015,11,-2.000,-0.650,-1.350,-0.183,1.603,Female hours down; male hours not up,Not yet assessed
6,Oman,2000,2008,8,-0.400,0.033,-0.433,1.778,0.666,Female hours down; male hours up,Not yet assessed
7,Poland,2004,2013,9,-0.367,0.016,-0.383,1.508,1.256,Female hours down; male hours up,Not yet assessed
8,Tanzania,2006,2014,8,-0.150,-0.083,-0.067,-7.030,5.469,Female hours down; male hours not up,Not yet assessed
9,Thailand,2009,2015,6,-0.227,0.083,-0.310,-3.591,0.119,Female hours down; male hours up,Not yet assessed


In [9]:
RESULTS_DIR = (
    PREPARED_DIR.parent
    / "analysis_results"
    / "time_changes"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "entity_history": entity_history,
    "adjacent_pair_changes": changes,
    "adjacent_change_directions": change_directions,
    "narrowing_decomposition": narrowing_decomposition,
    "adjacent_change_correlations": change_correlations,
    "first_last_changes": first_last,
    "first_last_correlations": first_last_correlations,
    "case_review": case_review,
}

for name, table in tables.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv", index=False)

print(f"Saved {len(tables)} result tables.")
print(f"Output directory: {RESULTS_DIR}")

Saved 8 result tables.
Output directory: D:\胡溪筼\Documents\2026-7Fall\Carolina Data Challenge 2026\Graduate_Dataset\analysis_results\time_changes
